# Predicting CONMEBOL World Cup qualifier scores

**DA380A Machine Learning · HT2026 · Group 28** — Lawrence Arciniegas, Gunnar Arias

**Goal:** predict the **home score and the away score** of international football matches from information known *before* kick-off, and evaluate the predictions on the **CONMEBOL World Cup qualifiers** of the 2026 cycle (the 10 South American national teams, 90 matches).

**Dataset:** Mart Jürisoo, *International football results from 1872* (Kaggle `martj42/international-football-results-from-1872-to-2017`; downloaded 13 Oct 2026 from the author's GitHub mirror — see the README).

**Algorithms:** (1) Linear Regression **implemented from scratch** with NumPy · (2) Random Forest Regressor (scikit-learn).

The notebook follows the 11-step structure of the project guidelines. The full plan is in [`docs/PLAN.md`](../docs/PLAN.md).

| Section | Step | Phase |
|---|---|---|
| 1 | Import libraries | 1 |
| 2 | Load the dataset | 1 |
| 3 | Explore the data | 1 |
| 4 | Preprocess the data (feature engineering) | 2 |
| 5 | Define features and target | 2 |
| 6 | Split into training and test sets | 4 |
| 7 | **Algorithm 1 — Linear Regression from scratch** | 3–4 |
| 8 | Algorithm 2 — Random Forest Regressor | 5 |
| 9 | Evaluate both models | 6 |
| 10 | Compare the results | 6 |
| 11 | Visualisations | 6 |

## 1. Import libraries

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import (CONMEBOL, RANDOM_STATE, TARGETS, TEST_END, TEST_START, TRAIN_START,
                      clean_results, load_former_names, load_results)

from src.features import (ELO_K, FEATURES, FORM_WINDOW, H2H_WINDOW, VENUE_ALTITUDE_M,
                          build_features)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 2. Load the dataset

`results.csv` has one row per match: date, home and away team, the final score (penalty shoot-outs not included), tournament, city, country, and whether the venue was neutral.

In [ ]:
raw = load_results()
former = load_former_names()
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all").T

## 3. Explore the data

### 3.1 Data quality

In [ ]:
teams = pd.unique(raw[["home_team", "away_team"]].values.ravel())
print(f"Date range: {raw['date'].min().date()} -> {raw['date'].max().date()}")
print(f"Teams: {len(teams)} | tournaments: {raw['tournament'].nunique()}")
print(f"Missing values: {raw.isna().sum().sum()}")
print(f"Fully duplicated rows: {raw.duplicated().sum()}")
print(f"Same date + same teams: {raw.duplicated(subset=['date', 'home_team', 'away_team']).sum()}")
print(f"Negative scores: {((raw['home_score'] < 0) | (raw['away_score'] < 0)).sum()}")
raw[raw.duplicated(subset=["date", "home_team", "away_team"], keep=False)]

In [ ]:
ax = raw.groupby(raw["date"].dt.year).size().plot(figsize=(12, 3.5))
ax.axvspan(TRAIN_START.year, TEST_START.year, color="tab:green", alpha=0.08, label="training window")
ax.axvspan(TEST_START.year, TEST_END.year, color="tab:red", alpha=0.12, label="test window")
ax.set(title="International matches per year", xlabel="", ylabel="Matches")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

The raw data is clean in the technical sense: no missing values, no duplicated rows, no impossible scores. The one pair with the same date and teams (Tahiti vs New Caledonia, 1974) has two *different* scores, so it is most likely two real matches on the same day; it is from before our training window and is kept.

### 3.2 Cleaning: which teams and matches we keep

1. **Former team names.** `former_names.csv` lists 36 old names (e.g. Dahomey → Benin). None of them occurs in `results.csv` — the dataset already uses one name per team — so nothing has to be renamed. We keep the check so it stays visible.
2. **Only FIFA national teams.** The data also contains regional and unrecognised teams (Catalonia, Corsica, Tibet, …) from tournaments such as the Island Games or the CONIFA World Football Cup. They produce extreme results (e.g. 22–0) that have nothing to do with our problem. We keep only teams that have played at least one **World Cup or World Cup qualifier** — a simple, data-driven definition of a FIFA member.
3. **Periods.** The full history is kept, because Phase 2 needs old matches to build team ratings, but every match gets a label: *warm-up* (before 1994, features only), *train* (1994 → Aug 2023), *test* (Sep 2023 → Sep 2025, the 2026 qualifying cycle) and *after*.

In [ ]:
df, notes = clean_results(raw, former)
print("\n".join(notes))

In [ ]:
removed = sorted(set(teams) - set(df["home_team"]) - set(df["away_team"]))
print(f"{len(removed)} teams removed, e.g.:", ", ".join(removed[:15]), "...")
df["period"].value_counts().reindex(["warm-up", "train", "test", "after"]).to_frame("matches")

> **From here on, the exploration only uses the training period** (plus the warm-up history where noted). The test period — including the 90 CONMEBOL qualifiers of the 2026 cycle — is not looked at until the final evaluation, so it can't influence our choices.

In [ ]:
train = df[df["period"] == "train"].copy()
train["total_goals"] = train["home_score"] + train["away_score"]
train["margin"] = train["home_score"] - train["away_score"]
print(f"Training matches: {len(train):,} | teams: {pd.unique(train[['home_team', 'away_team']].values.ravel()).size}")

### 3.3 The targets: home and away goals

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
bins = list(range(0, 10))
labels = [str(b) for b in bins[:-1]] + ["9+"]
counts = pd.DataFrame({
    side: train[col].clip(upper=9).value_counts(normalize=True).reindex(bins, fill_value=0).values
    for side, col in [("Home goals", "home_score"), ("Away goals", "away_score")]
}, index=labels)
counts.plot(kind="bar", ax=ax, rot=0, width=0.8)
ax.set(title="Goals per team in a match (training period)", xlabel="Goals", ylabel="Share of matches")
plt.tight_layout(); plt.show()

train[TARGETS].agg(["mean", "var", "median", "max"]).T.round(2)

In [ ]:
print("Matches won by 10 or more goals:", (train["margin"].abs() >= 10).sum())
train.loc[train["margin"].abs().sort_values(ascending=False).index[:5],
          ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]]

### 3.4 Home advantage

In [ ]:
def outcome_table(d):
    return pd.Series({
        "matches": len(d),
        "home goals": d["home_score"].mean(),
        "away goals": d["away_score"].mean(),
        "home win %": 100 * (d["margin"] > 0).mean(),
        "draw %": 100 * (d["margin"] == 0).mean(),
        "away win %": 100 * (d["margin"] < 0).mean(),
    })

venue = train.groupby(train["neutral"].map({False: "home venue", True: "neutral venue"})).apply(outcome_table)
ax = venue[["home win %", "draw %", "away win %"]].plot(kind="bar", rot=0, figsize=(7, 3.5))
ax.set(title="Results by venue (training period)", xlabel="", ylabel="% of matches")
plt.tight_layout(); plt.show()
venue.round(2)

### 3.5 Match importance

In [ ]:
importance = train.groupby("tournament_group", observed=True).apply(
    lambda d: pd.Series({"matches": len(d), "goals per match": d["total_goals"].mean(),
                         "mean |margin|": d["margin"].abs().mean(),
                         "home win %": 100 * (d["margin"] > 0).mean()}))
importance.round(2)

### 3.6 CONMEBOL World Cup qualifiers (training period)

Since the qualifiers for the 1998 World Cup, CONMEBOL plays one league: every team meets every other team home and away.

In [ ]:
wcq = train[train["is_conmebol_wcq"] & (train["date"] >= "1996-01-01")].copy()
cycle_end = {1997: 1998, 2001: 2002, 2005: 2006, 2009: 2010, 2013: 2014, 2017: 2018, 2022: 2022}
wcq["cycle"] = wcq["date"].dt.year.map(lambda y: next(wc for end, wc in cycle_end.items() if y <= end))
print(wcq.groupby("cycle").size().rename("matches").to_string())
print(f"\nAll CONMEBOL qualifiers 1996-2022: {len(wcq)}")
outcome_table(wcq).round(2)

In [ ]:
rows = []
for team in CONMEBOL:
    home = wcq[wcq["home_team"] == team]
    away = wcq[wcq["away_team"] == team]
    home_pts = 3 * (home["margin"] > 0) + (home["margin"] == 0)
    away_pts = 3 * (away["margin"] < 0) + (away["margin"] == 0)
    rows.append({"team": team, "played": len(home) + len(away),
                 "points per game, home": home_pts.mean(), "points per game, away": away_pts.mean(),
                 "goals for per game": (home["home_score"].sum() + away["away_score"].sum()) / (len(home) + len(away)),
                 "goals against per game": (home["away_score"].sum() + away["home_score"].sum()) / (len(home) + len(away))})
teams_wcq = pd.DataFrame(rows).set_index("team").sort_values("points per game, home", ascending=False)

ax = teams_wcq[["points per game, home", "points per game, away"]].plot(kind="bar", rot=0, figsize=(11, 3.8))
ax.set(title="CONMEBOL qualifiers 1996-2022: points per game at home and away", xlabel="", ylabel="Points per game")
plt.tight_layout(); plt.show()
teams_wcq.round(2)

### 3.7 Altitude

Bolivia plays its home qualifiers in La Paz (≈ 3,640 m) and recently El Alto (≈ 4,100 m); Ecuador mostly in Quito (≈ 2,850 m); Colombia sometimes in Bogotá (≈ 2,640 m). Do visiting teams struggle there?

In [ ]:
HIGH_ALTITUDE_M = {"La Paz": 3640, "El Alto": 4100, "Quito": 2850, "Bogotá": 2640}
wcq["venue"] = np.where(wcq["city"].isin(["La Paz", "El Alto"]), "Bolivia: La Paz / El Alto",
               np.where(wcq["city"].isin(["Quito", "Bogotá"]), "Quito / Bogotá", "other venues"))
altitude = wcq.groupby("venue").apply(outcome_table)
altitude.round(2)

In [ ]:
# Same team, different altitude: Bolivia and Ecuador at home vs away
for team in ["Bolivia", "Ecuador"]:
    print(f"{team}: points per game at home {teams_wcq.loc[team, 'points per game, home']:.2f}, "
          f"away {teams_wcq.loc[team, 'points per game, away']:.2f}")

### 3.8 Main observations

- **Data quality.** 49,945 matches from 1872 to Oct 2026 with no missing values, no duplicated rows and no impossible scores. Removing regional and unrecognised teams dropped 2,565 matches; the training period then has **25,266 matches between 211 national teams**. The dip in 2020 is the COVID-19 pandemic.
- **The targets are small, skewed counts.** Home teams score 1.62 goals on average and away teams 1.08, with a median of 1 for both. 26% of home teams and 40% of away teams score 0. The variance (2.73 and 1.65) is larger than the mean — a long tail of big wins. 101 training matches were won by 10 or more goals (e.g. Australia 31–0 American Samoa); they are real, official matches, so we keep them, but they will push up the RMSE of every model. Team-strength features should explain most of them.
- **Home advantage is large.** At a home venue the home team scores 1.67 goals against 1.00 and wins 50.8% of the matches (away wins 25.7%). On neutral ground the gap almost disappears (1.49 vs 1.29). → `neutral` must be a feature.
- **Match importance matters.** Qualifiers have the most lopsided results (mean margin 1.84 goals, against 1.37 in friendlies), because they include mismatches between strong and very weak teams. → a match-importance feature.
- **CONMEBOL qualifiers 1996–2022:** 593 matches in 7 cycles (72 when Brazil qualified automatically for 1998 and as hosts for 2014; 89 for 2022 because Brazil vs Argentina in September 2021 was suspended). The home team wins 54.0% (draw 24.3%, away win 21.7%) and scores 1.67 goals against 0.95 — a **stronger home advantage than in international football overall**.
- **Brazil and Argentina** are the strongest teams both at home and away (2.59 / 1.56 and 2.30 / 1.57 points per game).
- **Altitude.** Bolivia takes 1.59 points per game at home but only 0.13 away; Ecuador 2.13 against 0.72 — by far the biggest home/away gaps. In Quito and Bogotá the home team wins 59.7% and visitors score only 0.72 goals per game. In La Paz / El Alto the home win rate is lower (42.6%) because Bolivia is a weaker team overall, but compared with Bolivia's away record the effect is huge. → we move the **altitude** feature from "stretch" to the planned features of Phase 2.

**What this means for Phase 2:** team strength (Elo ratings) to explain the big wins and the dominance of Brazil and Argentina, recent form, the `neutral` flag, a match-importance feature, and venue altitude.

## 4. Preprocess the data — feature engineering

The raw table only says *who* played *where* and *what the score was*. A model needs to know, **before kick-off**, how strong both teams are, how they have played recently, how they did against each other, and where the match is played. All features are built in `src/features.py`:

| Feature | For each match | Why (from section 3) |
|---|---|---|
| `elo_home`, `elo_away` | Elo rating of both teams | Team strength — explains the big wins and the dominance of Brazil and Argentina |
| `form_gf_*`, `form_ga_*`, `form_pts_*` | Goals for, goals against and points per match over each team's last 10 matches | Current form, which ratings react to only slowly |
| `h2h_gd`, `h2h_n` | Mean goal difference (home-team view) in the last 5 meetings, and how many meetings there were | Some match-ups have a history of their own |
| `neutral` | 1 if the venue is neutral | Home advantage is ≈ 0.7 goals at a home venue and almost gone on neutral ground |
| `altitude_km` | Venue altitude for venues at 1,500 m or higher (La Paz 3.64 km, Quito 2.85 km, …), else 0 | Bolivia and Ecuador have the largest home/away gaps |
| `is_world_cup`, `is_continental_final`, `is_qualifier`, `is_other_tournament` | Tournament group (Friendly = reference) | Qualifiers have the most lopsided scores |
| `year` | Calendar year | Long-term trends in scoring |

> **The leakage rule.** Matches are processed in date order. For every match, the features are read **first**, and only **then** is the match's result added to the team statistics. So no match can see its own result or any later result. `tests/test_features.py` checks this: changing the score of one match must not change the features of that match or of any earlier match.

**Elo ratings** (World Football Elo method, eloratings.net). Every team starts at 1500. After each match:

$$R_\text{new} = R_\text{old} + K \cdot G \cdot (W - W_e), \qquad W_e = \frac{1}{10^{-dr/400} + 1}$$

- $W$ = 1 for a win, 0.5 for a draw, 0 for a loss; $W_e$ = the expected result, where $dr$ is the rating difference **+100 for the home team** (unless the venue is neutral).
- $K$ = match importance: 60 World Cup · 50 continental final · 40 qualifier / Nations League · 30 other tournament · 20 friendly.
- $G$ = margin of victory: 1 for 0–1 goals, 1.5 for 2 goals, $(11 + N)/8$ for $N \geq 3$.

The ratings are computed over the **whole history since 1872**, so they are already meaningful when the training period starts in 1994.

In [ ]:
feat = build_features(df)
data = df.drop(columns="neutral").join(feat)          # the features' 0/1 version of `neutral` is kept
print(f"Feature table: {len(feat):,} matches x {len(FEATURES)} model features (built in date order)")
data.loc[data["period"] == "train", ["date", "home_team", "away_team", "home_score", "away_score"] + FEATURES].tail(3)

### 4.1 Do the Elo ratings make sense?

In [ ]:
train_data = data[data["period"] == "train"]

# pre-match rating of every CONMEBOL team over the training period
long = pd.concat([
    train_data[["date", "home_team", "elo_home"]].set_axis(["date", "team", "elo"], axis=1),
    train_data[["date", "away_team", "elo_away"]].set_axis(["date", "team", "elo"], axis=1),
])
long = long[long["team"].isin(CONMEBOL)].sort_values("date")

fig, ax = plt.subplots(figsize=(12, 4.5))
for team, d in long.groupby("team"):
    ax.plot(d["date"], d["elo"].rolling(5, min_periods=1).mean(), label=team, lw=1.4)
ax.set(title="Elo rating of the CONMEBOL teams, 1994 – Aug 2023 (pre-match, smoothed over 5 matches)",
       xlabel="", ylabel="Elo rating")
ax.legend(ncol=5, fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

In [ ]:
# each team's pre-match rating in its last training match (end of August 2023)
last_home = train_data[["date", "home_team", "elo_home"]].set_axis(["date", "team", "elo"], axis=1)
last_away = train_data[["date", "away_team", "elo_away"]].set_axis(["date", "team", "elo"], axis=1)
latest = (pd.concat([last_home, last_away]).sort_values("date", kind="stable")
            .groupby("team").tail(1).set_index("team")["elo"].sort_values(ascending=False))
print("Top 10 at the end of the training period:")
print(latest.head(10).round(0).to_string())
print("\nCONMEBOL teams:")
print(latest[CONMEBOL].sort_values(ascending=False).round(0).to_string())

In [ ]:
bands = pd.cut(train_data["elo_diff"], [-2000, -200, -100, 0, 100, 200, 2000],
               labels=["< -200", "-200 to -100", "-100 to 0", "0 to 100", "100 to 200", "> 200"])
home_win = (train_data["home_score"] > train_data["away_score"]).groupby(bands, observed=True).mean() * 100
ax = home_win.plot(kind="bar", rot=0, figsize=(8, 3.5), color="tab:blue")
ax.set(title="Home win % by pre-match Elo difference (home − away), training period",
       xlabel="Elo difference", ylabel="Home win %")
for i, v in enumerate(home_win):
    ax.text(i, v + 1, f"{v:.0f}%", ha="center")
plt.tight_layout(); plt.show()

### 4.2 How strongly is each feature related to the targets?

In [ ]:
corr = pd.DataFrame({target: train_data[FEATURES + ["elo_diff"]].corrwith(train_data[target])
                     for target in TARGETS})
ax = corr.sort_values("home_score").plot(kind="barh", figsize=(8, 6))
ax.axvline(0, color="black", lw=0.8)
ax.set(title="Correlation with home and away goals (training period)", xlabel="Pearson correlation")
plt.tight_layout(); plt.show()
corr.round(3).sort_values("home_score", ascending=False)

### 4.3 Altitude

In [ ]:
at_home = train_data[train_data["neutral"] == 0]
high = at_home["altitude_km"] > 0
print(f"Training matches at a venue of 1,500 m or higher: {high.sum():,} "
      f"(test period: {(data.loc[data['period'] == 'test', 'altitude_km'] > 0).sum()})")
pd.DataFrame({
    "matches": [high.sum(), (~high).sum()],
    "home goals": [at_home.loc[high, "home_score"].mean(), at_home.loc[~high, "home_score"].mean()],
    "away goals": [at_home.loc[high, "away_score"].mean(), at_home.loc[~high, "away_score"].mean()],
    "home win %": [100 * (at_home.loc[high, "home_score"] > at_home.loc[high, "away_score"]).mean(),
                   100 * (at_home.loc[~high, "home_score"] > at_home.loc[~high, "away_score"]).mean()],
}, index=["venue ≥ 1,500 m", "lower venues"]).round(2)

### 4.4 Missing values in the features

In [ ]:
missing = data.loc[data["period"].isin(["train", "test"]), FEATURES].isna()
print("Missing values per period:")
print(missing.groupby(data["period"], observed=True).sum().loc[:, lambda d: d.sum() > 0])
data.loc[missing.index[missing.any(axis=1)], ["date", "home_team", "away_team", "period", "n_prev_home", "n_prev_away"]]

### 4.5 Feature observations

- **The Elo ratings behave as they should.** The home team wins 14% of the time when it is rated more than 200 points lower, and 81% when it is rated more than 200 points higher, with a steady rise in between. At the end of the training period (Aug 2023) the top three are Argentina (2202), Brazil (2180) and France (2163) — the 2022 world champion and the two other strongest teams of that time. Among the CONMEBOL teams Bolivia is lowest (1677), and the chart shows known stories such as Venezuela's rise from about 1,300 around 2000 to about 1,800.
- **The strongest single signal is the rating difference** (r = 0.46 with home goals, −0.42 with away goals), followed by the head-to-head record (0.36 / −0.32) and how many goals the *opponent* has been conceding (0.35 / 0.32). `elo_diff` itself is not a model feature, because it is exactly `elo_home − elo_away` and would make the normal equation unsolvable; a linear model can rebuild it from the two ratings.
- **`neutral`, the tournament dummies, `year` and `altitude_km` are hardly correlated with the goals on their own.** That doesn't make them useless: home advantage, for example, works *together* with team strength. Whether they help will show in the models (sections 7–10).
- **Altitude:** 675 training matches were played at venues of 1,500 m or higher. There, away teams score 0.85 goals per match against 1.01 elsewhere, and the home team wins 53.3% against 50.7%. Only 4% of matches are at altitude, which is why the overall correlation is close to 0.
- **Missing values:** 9 training matches have no recent form, because they are the **first matches of new national teams** (e.g. Czech Republic 1994, Kosovo 2014, South Sudan 2012). The test period has none. They are filled with training medians in section 6. A related limitation: successor states such as the Czech Republic start at Elo 1500 instead of inheriting their predecessor's rating.

## 5. Define features and target

- **Targets (y):** `home_score` and `away_score` — two numbers per match. Both algorithms predict them together (our scratch model fits one coefficient column per target; the random forest supports several targets natively).
- **Features (X):** the 17 pre-match features from section 4. They are all numeric already, so no encoding is needed; scaling and filling the few missing form values happen in section 6, fitted on the training data only.

In [ ]:
FEATURE_DESCRIPTIONS = {
    "elo_home": "Elo rating of the home team before the match",
    "elo_away": "Elo rating of the away team before the match",
    "form_gf_home": f"Home team: goals scored per match, last {FORM_WINDOW} matches",
    "form_ga_home": f"Home team: goals conceded per match, last {FORM_WINDOW} matches",
    "form_pts_home": f"Home team: points per match (3/1/0), last {FORM_WINDOW} matches",
    "form_gf_away": f"Away team: goals scored per match, last {FORM_WINDOW} matches",
    "form_ga_away": f"Away team: goals conceded per match, last {FORM_WINDOW} matches",
    "form_pts_away": f"Away team: points per match (3/1/0), last {FORM_WINDOW} matches",
    "h2h_gd": f"Mean goal difference (home view) in the last {H2H_WINDOW} meetings, 0 if none",
    "h2h_n": f"Number of earlier meetings counted (0-{H2H_WINDOW})",
    "neutral": "1 if played on neutral ground",
    "altitude_km": "Venue altitude in km for venues at 1,500 m or higher, else 0",
    "is_world_cup": "1 if a World Cup match",
    "is_continental_final": "1 if a continental championship match (Copa América, Euro, ...)",
    "is_qualifier": "1 if a qualifier or Nations League match",
    "is_other_tournament": "1 if another tournament (friendly = all four are 0)",
    "year": "Calendar year of the match",
}
assert list(FEATURE_DESCRIPTIONS) == FEATURES

X = data[FEATURES]
y = data[TARGETS]
print("X:", X.shape, "| y:", y.shape)
pd.Series(FEATURE_DESCRIPTIONS, name="meaning").to_frame()

In [ ]:
# save the feature table so later phases can start from it without rebuilding (git-ignored)
out = ROOT / "data" / "processed" / "features.csv"
data.to_csv(out, index=False)
print(f"Saved {out.relative_to(ROOT)} ({len(data):,} rows)")

## 6. Split into training and test sets

_To do — Phase 4: chronological split (train < 1 Sep 2023 ≤ test)._

## 7. Algorithm 1 — Linear Regression from scratch

_To do — Phases 3–4: `LinearRegressionScratch` (normal equation + gradient descent + L2)._

## 8. Algorithm 2 — Random Forest Regressor

_To do — Phase 5: grid search with `TimeSeriesSplit`._

## 9. Evaluate both models

_To do — Phase 6: MAE, MSE, RMSE, R² against the baselines._

## 10. Compare the results

_To do — Phase 6._

## 11. Visualisations

_To do — Phase 6._